# Import Library

In [1]:
import pandas as pd
import os
import opendatasets as od
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
import random

In [2]:
# set random seed
seed=3
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Load Data

In [37]:
# Prepare dataset
def list_all_dataset(dir: str):

    label_name=[]
    cat_ids=[]
    breeds=[]
    sexes=[]
    owner_ids=[]
    recording_sessions=[]
    vocalization_counters=[]
    audio_name=[]
    audio_length=[]
    amplitude=[]
    amplitude_avg=[]
    sampling_rate=[]
    file_path=[]

    files = sorted(os.listdir(dir))
    for _, file in enumerate(files):
        if file.endswith(".wav"):
            additional_info = file.split("_")[-1].split(".")[0]
            label, cat_id, breed, sex, owner_id, recording_session, vocalization_counter = file.split("_")[:-1] + [str(int(additional_info[0])), str(int(additional_info[1:]))]
            audio_path = os.path.join(dir, file)
            
            label_name.append(label)
            cat_ids.append(cat_id)
            breeds.append(breed)
            sexes.append(sex)
            owner_ids.append(owner_id)
            recording_sessions.append(recording_session)
            vocalization_counters.append(vocalization_counter)
            audio_name.append(file)

            length = librosa.get_duration(filename=audio_path)
            y, sr = librosa.load(audio_path)

            amplitude.append(y)
            amplitude_avg.append(y.mean())
            sampling_rate.append(sr)
            audio_length.append(length)
            file_path.append(audio_path)

    df = pd.DataFrame({
        "label":label_name,
        "cat_ids":cat_ids,
        "breeds":breeds,
        "sexes":sexes,
        "owner_ids":owner_ids,
        "recording_sessions":recording_sessions,
        "vocalization_counters":vocalization_counters,
        "audio":audio_name,
        "length":audio_length,
        "amplitude":amplitude,
        "amplitude_avg":amplitude_avg,
        "sr":sampling_rate,
        "path":audio_path
    }).sort_values(['label','cat_ids','breeds','sexes','owner_ids','recording_sessions','vocalization_counters']).reset_index(drop=True)

    return df

In [33]:
dataset = 'https://www.kaggle.com/datasets/andrewmvd/cat-meow-classification'
od.download(dataset)

Skipping, found downloaded files in "./cat-meow-classification" (use force=True to force download)


In [38]:
dir = '/Users/fadilahnurimani/Documents/Projects/cat-meow-audio-classifier/cat-meow-classification/dataset/dataset'
df = list_all_dataset(dir)

/var/folders/q_/5v_py9t567xctvklp6nv2ck40000gn/T/ipykernel_35073/1996682589.py:34: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  length = librosa.get_duration(filename=audio_path)


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   label                  440 non-null    object 
 1   cat_ids                440 non-null    object 
 2   breeds                 440 non-null    object 
 3   sexes                  440 non-null    object 
 4   owner_ids              440 non-null    object 
 5   recording_sessions     440 non-null    object 
 6   vocalization_counters  440 non-null    object 
 7   audio                  440 non-null    object 
 8   length                 440 non-null    float64
 9   amplitude              440 non-null    object 
 10  amplitude_avg          440 non-null    float32
 11  sr                     440 non-null    int64  
 12  path                   440 non-null    object 
dtypes: float32(1), float64(1), int64(1), object(10)
memory usage: 43.1+ KB


In [36]:
df.head()

,label,cat_ids,breeds,sexes,owner_ids,recording_sessions,vocalization_counters,audio,length,amplitude,amplitude_avg,sr,path
0,B,ANI01,MC,FN,SIM01,1,1,B_ANI01_MC_FN_SIM01_101.wav,2.281500,"[-9.183165e-08, -2.537417e-07, -1.84841e-07, 1...",-0.000440,22050,/Users/fadilahnurimani/Documents/Projects/cat-...
1,B,ANI01,MC,FN,SIM01,1,2,B_ANI01_MC_FN_SIM01_102.wav,1.419000,"[0.00020511363, 0.00016822206, -3.258395e-05, ...",0.000004,22050,/Users/fadilahnurimani/Documents/Projects/cat-...
2,B,ANI01,MC,FN,SIM01,1,3,B_ANI01_MC_FN_SIM01_103.wav,1.798875,"[0.00023735344, -6.777141e-05, -0.0005998552, ...",0.000012,22050,/Users/fadilahnurimani/Documents/Projects/cat-...
3,B,ANI01,MC,FN,SIM01,3,1,B_ANI01_MC_FN_SIM01_301.wav,1.739500,"[-3.3097282e-07, -3.4070115e-07, 2.0664572e-08...",0.000103,22050,/Users/fadilahnurimani/Documents/Projects/cat-...
4,B,ANI01,MC,FN,SIM01,3,2,B_ANI01_MC_FN_SIM01_302.wav,1.268000,"[0.00013297691, 0.00018529932, 0.00015624601, ...",-0.000021,22050,/Users/fadilahnurimani/Documents/Projects/cat-...


# Preprocessing Functions

# EDA